# Merge flag_3 (13 Oktober) ke cleaning.db
Alur: backup DB → tambah kolom `flag_3_new` (kosong) → baca CSV → update/insert per baris → `flag_3_new = 1` untuk semua baris CSV.

In [1]:
import sqlite3, shutil
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
DB_PATH = (BASE_DIR / "cleaning.db").resolve()
CSV_PATH = BASE_DIR / "cleaning_usaha_flag_3_13_oktober.csv"  # sesuaikan lokasi CSV

TABEL = "usaha"
KEYS = ["assignment_id", "nama_usaha"]
FLAG_COL = "flag_3_new"
STATUS_COL = "status_penyelesaian"
KET_COL = "keterangan"
UPDATED_COL = "updated_at"

assert DB_PATH.exists(), f"Database tidak ditemukan: {DB_PATH}"
assert CSV_PATH.exists(), f"CSV tidak ditemukan: {CSV_PATH}"
print("DB :", DB_PATH)
print("CSV:", CSV_PATH)


DB : D:\2026\Sensus Ekonomi\cleaning_data\3 Oktober 2026\Mikro\RantabDBNAS\cleaning.db
CSV: d:\2026\Sensus Ekonomi\cleaning_data\3 Oktober 2026\Mikro\RantabDBNAS\cleaning_usaha_flag_3_13_oktober.csv


In [3]:
try:
    conn.close()
except Exception:
    pass

shutil.copy2(backup, DB_PATH)
print("DB dikembalikan dari:", backup)

DB dikembalikan dari: D:\2026\Sensus Ekonomi\cleaning_data\3 Oktober 2026\Mikro\RantabDBNAS\cleaning_backup_sebelum_flag3.db


In [4]:
# Backup dulu (sekali saja), lalu buka DB
backup = DB_PATH.with_name(DB_PATH.stem + "_backup_sebelum_flag3.db")
if not backup.exists():
    shutil.copy2(DB_PATH, backup)
    print("Backup dibuat:", backup)
else:
    print("Backup sudah ada:", backup)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

info = cur.execute(f'PRAGMA table_info("{TABEL}")').fetchall()
db_cols = [r[1] for r in info]
pk = [r for r in info if r[5] > 0]
pk_cols = [r[1] for r in pk] if len(pk) == 1 and (pk[0][2] or "").upper() == "INTEGER" else []

# level_4_full_code adalah kolom SUMBER dari CSV, tidak wajib ada di DB.
# level_2_code wajib ada di DB karena akan disimpan ke sana.
wajib_db = [c for c in KEYS + [STATUS_COL, KET_COL, "level_2_code"] if c not in db_cols]
if wajib_db:
    raise RuntimeError(
        f"Kolom yang dibutuhkan tidak ditemukan di tabel {TABEL}: {wajib_db}. "
        "Catatan: level_4_full_code tidak wajib ada di DB; kolom itu dibaca dari CSV."
    )

# Tambah kolom flag_3_new (kosong / NULL) bila belum tersedia
if FLAG_COL not in db_cols:
    cur.execute(f'ALTER TABLE "{TABEL}" ADD COLUMN "{FLAG_COL}" INTEGER')
    conn.commit()
    db_cols.append(FLAG_COL)
    print(f"Kolom {FLAG_COL} ditambahkan")

# Guard: jangan jalan 2x tanpa memeriksa/restore backup
sudah = cur.execute(
    f'SELECT COUNT(*) FROM "{TABEL}" WHERE "{FLAG_COL}" IS NOT NULL'
).fetchone()[0]
if sudah:
    raise RuntimeError(
        f"{FLAG_COL} sudah terisi di {sudah} baris. "
        "Proses dihentikan agar tidak menggandakan/mengubah data. "
        "Jangan hapus flag secara manual; periksa hasil proses atau pulihkan backup."
    )

print("Total baris DB:", cur.execute(f'SELECT COUNT(*) FROM "{TABEL}"').fetchone()[0])
print("PK autoincrement:", pk_cols)
print("updated_at tersedia di DB:", UPDATED_COL in db_cols)
print("level_2_code tersedia di DB:", "level_2_code" in db_cols)


Backup sudah ada: D:\2026\Sensus Ekonomi\cleaning_data\3 Oktober 2026\Mikro\RantabDBNAS\cleaning_backup_sebelum_flag3.db
Kolom flag_3_new ditambahkan
Total baris DB: 1850931
PK autoincrement: []
updated_at tersedia di DB: False
level_2_code tersedia di DB: True


In [5]:
df = pd.read_csv(CSV_PATH, dtype=str)
# Rapikan spasi pada nama kolom CSV agar pencocokan dengan kolom DB konsisten
df.columns = df.columns.str.strip()

# level_2_code = 4 digit pertama dari level_4_full_code.
# Jika level_4_full_code kosong/tidak tersedia, gunakan level_6_full_code sebagai cadangan.
def bersihkan_kode(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

l4 = bersihkan_kode(df["level_4_full_code"]) if "level_4_full_code" in df.columns else pd.Series("", index=df.index, dtype="string")
l6 = bersihkan_kode(df["level_6_full_code"]) if "level_6_full_code" in df.columns else pd.Series("", index=df.index, dtype="string")
kode_sumber = l4.where(l4 != "", l6)
df["level_2_code"] = kode_sumber.str[:4].where(kode_sumber.str.len() >= 4, None)

print("Ukuran CSV:", df.shape)
print("level_2_code kosong di CSV:", int(df["level_2_code"].isna().sum()))
print("Contoh pembentukan kode:")
display(df[[c for c in ["level_4_full_code", "level_6_full_code", "level_2_code"] if c in df.columns]].head(10))

for k in KEYS:
    assert k in df.columns, f"Kolom key '{k}' tidak ada di CSV"

# Pastikan kolom yang akan ditulis memang tersedia di DB.
skip_cols = [c for c in df.columns if c not in db_cols]
if skip_cols:
    print("PERINGATAN - kolom CSV tidak ada di DB (dilewati):", skip_cols)

special = {STATUS_COL, KET_COL, FLAG_COL}
data_cols = [c for c in df.columns if c in db_cols and c not in pk_cols and c not in special]
if "level_2_code" not in data_cols:
    raise RuntimeError(
        "level_2_code tidak masuk data_cols. "
        f"Kolom DB tersedia: {[c for c in db_cols if 'level' in c.lower()]}; "
        f"Kolom CSV tersedia: {[c for c in df.columns if 'level' in c.lower()]}"
    )

print("level_2_code ikut diproses:", "level_2_code" in data_cols)
print("Kolom yang akan ditulis:", len(data_cols))
df.head()


Ukuran CSV: (344659, 25)
level_2_code kosong di CSV: 0
Contoh pembentukan kode:


,level_4_full_code,level_2_code
0,1602022006,1602
1,1608010001,1608
2,1671020013,1671
3,1603033006,1603
4,1673020007,1673
5,1609010011,1609
6,1611030008,1611
7,1671062003,1671
8,1605090006,1605
9,1610052005,1610


PERINGATAN - kolom CSV tidak ada di DB (dilewati): ['level_4_full_code', 'is_remark_diperbaiki', 'is_remark_sesuai_lapangan', 'link_fasih_sm']
level_2_code ikut diproses: True
Kolom yang akan ditulis: 21


,assignment_id,level_4_full_code,nama_usaha,kategori,kode_kbli,metrik_tenaga_kerja,metrik_upah_gaji,metrik_biaya_produksi,metrik_biaya_pembelian,metrik_biaya_operasional,...,metrik_total_aset,metrik_output,rasio_biaya_pembelian_barang_terhadap_omzet,rasio_ntb,produktivitas,flag_rasio_3_ntb_output,is_remark_diperbaiki,is_remark_sesuai_lapangan,link_fasih_sm,level_2_code
0,000009a7-fe00-41e5-bee2-0cc07cc65dc4,1602022006,USAHA AIR REBUSAN <NURHIDAYAH>,E,36001,1.0,0.0,500000.0,0.0,0.0,...,600000.0,1100000.0,0.0,0.545455,1100000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1602
1,00000ee5-68a2-4eb2-ac93-2234dc24489d,1608010001,WARUNG SEMBAKO (RONAL),G,47112,1.0,0.0,200000.0,10000000.0,200000.0,...,7000000.0,4000000.0,0.714286,0.9,4000000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1608
2,00004c39-e36d-4e08-8acc-4270d009ce67,1671020013,WARUNG HODIJAH,G,47112,1.0,0.0,420000.0,8400000.0,650000.0,...,3400000.0,19070000.0,0.305788,0.943891,19070000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1671
3,00010432-0be4-40b5-be6a-ebc6c05070e8,1603033006,Warung manisan (jajang suparman),G,47112,2.0,0.0,1200000.0,360000000.0,4000000.0,...,60000000.0,54000000.0,0.869565,0.903704,27000000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1603
4,00012661-ab08-436b-af8a-75537385a0a9,1673020007,Konter pulsa,G,47901,3.0,700000.0,30000000.0,0.0,200000.0,...,22000000.0,45000000.0,0.0,0.328889,15000000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1673


In [6]:
# Index supaya pencarian per baris tidak full-scan
key_sql = ", ".join(f'"{k}"' for k in KEYS)
cur.execute(f'CREATE INDEX IF NOT EXISTS idx_keys_flag3 ON "{TABEL}" ({key_sql})')
conn.commit()

where_sql = " AND ".join(f'"{k}" IS ?' for k in KEYS)
updated_expr = f'"{UPDATED_COL}"' if UPDATED_COL in db_cols else "NULL"

n_update = n_insert = n_pernah = 0

try:
    for _, rec in df.iterrows():
        row = {c: (None if pd.isna(rec[c]) else str(rec[c])) for c in df.columns}
        # Pastikan kode yang disimpan tidak berupa string kosong/spasi
        if row.get("level_2_code") is not None:
            row["level_2_code"] = row["level_2_code"].strip() or None

        key_vals = [row[k] for k in KEYS]

        found = cur.execute(
            f'SELECT rowid, "{STATUS_COL}", {updated_expr} FROM "{TABEL}" WHERE {where_sql}',
            key_vals,
        ).fetchall()

        if found:
            for rid, old_status, old_updated in found:
                # status selalu jadi 0; keterangan hanya diisi kalau sebelumnya 1
                if str(old_status).strip() in ("1", "1.0"):
                    ket = (
                        f"pernah diselesaikan pada tanggal {old_updated} "
                        "tapi belum ada remark di fasih"
                    )
                    n_pernah += 1
                else:
                    ket = None

                set_cols = data_cols + [STATUS_COL, KET_COL, FLAG_COL]
                vals = [row.get(c) for c in data_cols] + [0, ket, 1]
                set_sql = ", ".join(f'"{c}" = ?' for c in set_cols)
                cur.execute(
                    f'UPDATE "{TABEL}" SET {set_sql} WHERE rowid = ?',
                    vals + [rid],
                )
                n_update += 1
        else:
            ins_cols = data_cols + [STATUS_COL, KET_COL, FLAG_COL]
            vals = [row.get(c) for c in data_cols] + [0, None, 1]
            cols_sql = ", ".join(f'"{c}"' for c in ins_cols)
            ph = ", ".join("?" for _ in ins_cols)
            cur.execute(
                f'INSERT INTO "{TABEL}" ({cols_sql}) VALUES ({ph})',
                vals,
            )
            n_insert += 1

    conn.commit()
    print(
        f"Selesai. Update: {n_update} | Insert baru: {n_insert} | "
        f"Dapat keterangan 'pernah diselesaikan': {n_pernah}"
    )
except Exception:
    conn.rollback()
    print("Error - perubahan pada transaksi ini di-rollback")
    raise


Selesai. Update: 270640 | Insert baru: 74258 | Dapat keterangan 'pernah diselesaikan': 1486


In [8]:
# Verifikasi
keys_q = ", ".join(f'"{k}"' for k in KEYS)

print("flag_3_new = 1 :", cur.execute(f'SELECT COUNT(*) FROM "{TABEL}" WHERE "{FLAG_COL}" = 1').fetchone()[0],
      "| baris CSV:", len(df), "| key unik CSV:", len(df.drop_duplicates(KEYS)))
print("Total baris DB :", cur.execute(f'SELECT COUNT(*) FROM "{TABEL}"').fetchone()[0])

dup_db = pd.read_sql(
    f'SELECT {keys_q}, COUNT(*) AS jml FROM "{TABEL}" WHERE "{FLAG_COL}" = 1 '
    f'GROUP BY {keys_q} HAVING COUNT(*) > 1', conn)
print("Key duplikat di DB (flag=1):", len(dup_db))

display(pd.read_sql(
    f'SELECT {keys_q}, "{STATUS_COL}", "{KET_COL}", "{FLAG_COL}" '
    f'FROM "{TABEL}" WHERE "{FLAG_COL}" = 1 AND "{KET_COL}" IS NOT NULL LIMIT 10', conn))
conn.close()

flag_3_new = 1 : 344719 | baris CSV: 344659 | key unik CSV: 344548
Total baris DB : 1925189
Key duplikat di DB (flag=1): 160


,assignment_id,nama_usaha,status_penyelesaian,keterangan,flag_3_new
0,7602417a-ab92-4ca2-8f21-a296886198f1,UTP PETERNAKAN ASNAWI,0,pernah diselesaikan pada tanggal None tapi bel...,1
1,a369f7ba-c388-47a4-84dd-5b024891005e,UTP PERTERNAKAN HENDRI YANTO,0,pernah diselesaikan pada tanggal None tapi bel...,1
2,a63ee437-22b0-4932-8071-911cf9f6d707,JUAL PASIR BANGUNAN (PAK SUKRI),0,pernah diselesaikan pada tanggal None tapi bel...,1
3,a9aad3cd-fe54-4602-8a6f-182de3c14131,PENCARI TIMAH ( TAUFIQ ),0,pernah diselesaikan pada tanggal None tapi bel...,1
4,5aa47a7c-3f4c-4583-9a9a-92310ef7108a,PENGASAH NGASAH BATU CIN CIN,0,pernah diselesaikan pada tanggal None tapi bel...,1
5,682d6a64-2b84-4f6e-9984-5eee4d226b7e,USAHA PENGGALIAN BATU (NIRWANAH),0,pernah diselesaikan pada tanggal None tapi bel...,1
6,4419f709-df9a-4245-b181-0ec57edebc24,JUAL GALON,0,pernah diselesaikan pada tanggal None tapi bel...,1
7,e0f5096c-d6dc-46c5-8b40-ce89af88d352,ES TEH (LELI PARTISIA),0,pernah diselesaikan pada tanggal None tapi bel...,1
8,9e583fbf-710c-4788-9799-4458c2b9032b,PENJAHIT WIWIN,0,pernah diselesaikan pada tanggal None tapi bel...,1
9,8acb6f97-7e6f-4361-b86e-a08b9198ac93,PANGKAS RAMBUT DASS,0,pernah diselesaikan pada tanggal None tapi bel...,1


In [9]:
# Head DB (koneksi sendiri, bisa dijalankan kapan saja)
pd.set_option("display.max_columns", None)
with sqlite3.connect(DB_PATH) as c:
    df_db = pd.read_sql(f'SELECT * FROM "{TABEL}" LIMIT 5', c)
display(df_db)

,link,assignment_id,level_2_code,kategori,kode_kbli,nama_usaha,metrik_tenaga_kerja,metrik_upah_gaji,metrik_biaya_produksi,metrik_biaya_pembelian,metrik_biaya_operasional,metrik_biaya_non_operasional,metrik_total_pengeluaran,metrik_pendapatan_barang_jasa,metrik_pendapatan_lainnya,metrik_nilai_tambah,metrik_total_aset,metrik_output,rasio_biaya_pembelian_barang_terhadap_omzet,reklasifikasi_skala_usaha,rasio_ntb,produktivitas,flag_rasio_1_output_aset,flag_rasio_2_upah_ntb,flag_rasio_3_ntb_output,flag_kategori_P_dan_U,sumber_file,status_penyelesaian,keterangan,update_at,flag_3_new
0,https://fasih-sm.bps.go.id/app/assignment/fd68...,0067eab8-97e4-4169-940c-6b7d124b102f,1601,A,01111,PERKEBUNAN JAGUNG MARDANI,1,0,35000000,0,8000000,0,43000000,86000000,0,43000000,5000000,86000000,0.0,Mikro,0.500000,8.600000e+07,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,2026-10-06 11:07:20,None
1,https://fasih-sm.bps.go.id/app/assignment/fd68...,00754e8b-9421-4536-ac01-778f0ee3b0c4,1601,A,01111,UTP TANAMAN PANGAN (ISMANTO),2,0,14880000,0,8160000,0,23040000,0,0,-23040000,0,0,NaN,Mikro,NaN,0.000000e+00,None,1,None,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,1,Sesuai kondisi,2026-10-06 20:12:37,None
2,https://fasih-sm.bps.go.id/app/assignment/fd68...,008baf68-cf1b-44a5-9015-3ef04664306e,1601,A,01111,UTP TANAMAN PANGAN JAGUNG (MARKUS WARJU),7,5000000,7080000,0,1500000,0,13580000,59000000,0,50420000,115000000,59000000,0.0,Mikro,0.854576,8.428571e+06,1,1,0,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,None
3,https://fasih-sm.bps.go.id/app/assignment/fd68...,00a48eda-54fa-427f-8fcb-a1ebf42440bf,1601,A,01111,PERKEBUNAN JAGUNG (ASMARA),1,0,4000000,0,3000000,500000,7500000,12100000,0,5100000,0,12100000,0.0,Mikro,0.421488,1.210000e+07,None,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,None
4,https://fasih-sm.bps.go.id/app/assignment/fd68...,00e1b203-ea5a-4c97-832c-221264a478a2,1601,A,01111,PERKEBUNAN JAGUNG,2,0,24000000,0,6000000,2000000,32000000,45000000,0,15000000,1000000,45000000,0.0,Mikro,0.333333,2.250000e+07,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,None


In [10]:
lvl = "level_2_code"
tes = "1601"

csv_lvl = df[df[lvl] == tes]
print(
    f"CSV level {tes}  : {len(csv_lvl)} baris | key unik: {len(csv_lvl.drop_duplicates(KEYS))}"
)

with sqlite3.connect(DB_PATH) as c:
    n_db = c.execute(
        f'SELECT COUNT(*) FROM "{TABEL}" WHERE "{lvl}" = ? AND "{FLAG_COL}" = 1', (tes,)
    ).fetchone()[0]
print(f"DB flag=1 level {tes}: {n_db}")

# Perbandingan semua level_2_code
a = df.groupby(lvl).size().rename("csv_baris")
b = df.drop_duplicates(KEYS).groupby(lvl).size().rename("csv_key_unik")
with sqlite3.connect(DB_PATH) as c:
    d = pd.read_sql(
        f'SELECT "{lvl}", COUNT(*) AS db_flag1 FROM "{TABEL}" WHERE "{FLAG_COL}" = 1 GROUP BY "{lvl}"',
        c,
    ).set_index(lvl)["db_flag1"]

cmp = pd.concat([a, b, d], axis=1).fillna(0).astype(int)
cmp["selisih"] = cmp["csv_baris"] - cmp["db_flag1"]
display(cmp[cmp["selisih"] != 0].sort_values("selisih", ascending=False))

# Contoh key yang kembar di CSV
dup = df[df.duplicated(KEYS, keep=False)].sort_values(KEYS)
print("Baris CSV dengan key kembar:", len(dup))
display(dup[KEYS + ["level_4_full_code"]].head(20))

CSV level 1601  : 15882 baris | key unik: 15879
DB flag=1 level 1601: 15883


,csv_baris,csv_key_unik,db_flag1,selisih
level_2_code,,,,
1674,13211,13199,13207,4
1610,21374,21369,21372,2
1602,34053,34038,34052,1
1601,15882,15879,15883,-1
1606,23679,23675,23680,-1
1603,19085,19081,19088,-3
1605,13843,13843,13846,-3
1671,73835,73800,73839,-4
1613,7221,7216,7226,-5


Baris CSV dengan key kembar: 220


,assignment_id,nama_usaha,level_4_full_code
3495,028d039b-8f53-4e72-8be1-9486454ebc76,TOKO ALESIA (JONI SAPUTRA),1609061016
3496,028d039b-8f53-4e72-8be1-9486454ebc76,TOKO ALESIA (JONI SAPUTRA),1609061016
4535,0357d7ef-2a9c-4136-b720-81deb119c93a,JUAL SAYUR KELILING,1610060029
4536,0357d7ef-2a9c-4136-b720-81deb119c93a,JUAL SAYUR KELILING,1610060029
5016,03a7979e-5dc5-4528-8184-7ef12e1777ed,"Sang Hyang Seri, PT",1609060025
5017,03a7979e-5dc5-4528-8184-7ef12e1777ed,"Sang Hyang Seri, PT",1609060025
7166,054e1220-eb37-4cb6-ae83-b1a3d0eba047,SUPIR ANGKUTAN BARANG,1607040014
7167,054e1220-eb37-4cb6-ae83-b1a3d0eba047,SUPIR ANGKUTAN BARANG,1607040014
7446,0586c9bc-ad08-438a-92ca-bc163a7c088a,OJEK ONLINE,1671011002
7447,0586c9bc-ad08-438a-92ca-bc163a7c088a,OJEK ONLINE,1671011002


In [ ]:
# OPSIONAL: perbaiki level_2_code yang NULL untuk baris yang ditandai pada proses ini.
# Jalankan hanya setelah memeriksa hasil merge; tidak mengubah flag atau status.
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

sebelum = cur.execute(
    f'SELECT COUNT(*) FROM "{TABEL}" WHERE "{FLAG_COL}" = 1 AND "level_2_code" IS NULL'
).fetchone()[0]
print("level_2_code NULL (flag=1) sebelum:", sebelum)

ada = df[df["level_2_code"].notna()].copy()
cur.executemany(
    f'UPDATE "{TABEL}" SET "level_2_code" = ? '
    f'WHERE "{KEYS[0]}" IS ? AND "{KEYS[1]}" IS ? '
    f'AND ("level_2_code" IS NULL OR TRIM("level_2_code") = "")',
    [
        (str(r["level_2_code"]).strip(), r[KEYS[0]], r[KEYS[1]])
        for _, r in ada.iterrows()
    ],
)
conn.commit()

sesudah = cur.execute(
    f'SELECT COUNT(*) FROM "{TABEL}" WHERE "{FLAG_COL}" = 1 AND "level_2_code" IS NULL'
).fetchone()[0]
print("level_2_code NULL (flag=1) sesudah:", sesudah)
conn.close()


In [11]:
with sqlite3.connect(DB_PATH) as conn:
    query = """
        SELECT
            COALESCE(NULLIF(TRIM(level_2_code), ''), '(KOSONG)') AS level_2_code,
            COUNT(*) AS jumlah_data
        FROM usaha
        WHERE flag_3_new = 1
        GROUP BY COALESCE(NULLIF(TRIM(level_2_code), ''), '(KOSONG)')
        ORDER BY level_2_code;
    """

    rekap = pd.read_sql_query(query, conn)

display(rekap)

,level_2_code,jumlah_data
0,1601,15883
1,1602,34052
2,1603,19088
3,1604,20700
4,1605,13846
5,1606,23680
6,1607,25759
7,1608,13271
8,1609,31833
9,1610,21372
